# KPI Calculations

## Imports

In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

## Dataset

In [2]:
base_ordner = Path().resolve().parent
proc_data_ordner = base_ordner / "data" / "processed"

df_l = pd.read_csv(proc_data_ordner / "lauschke_gmbh.csv")
df_bench = pd.read_csv(proc_data_ordner / "benchmark.csv")

## Intensity KPIs
CO2, Energy, Water normalized for Revenue

In [3]:
df_l["CO2_intensity"] = (df_l["CarbonEmissions"] / df_l["Revenue"]).round(2)
df_l["energy_intensity"] = (df_l["EnergyConsumption"] / df_l["Revenue"]).round(2)
df_l["water_intensity"] = (df_l["WaterUsage"] / df_l["Revenue"]).round(2)

df_l[["Year", "Revenue", "CarbonEmissions", "CO2_intensity", "EnergyConsumption", "energy_intensity", "WaterUsage", "water_intensity"]]

,Year,Revenue,CarbonEmissions,CO2_intensity,EnergyConsumption,energy_intensity,WaterUsage,water_intensity
0,2015,2030.0,569254.8,280.42,9487579.2,4673.68,759006.3,373.89
1,2016,2008.7,551293.1,274.45,9188218.8,4574.21,735057.5,365.94
2,2017,1852.9,514255.6,277.54,8570927.0,4625.68,685674.2,370.05
3,2018,1567.0,433770.3,276.82,7229504.9,4613.60,578360.4,369.09
4,2019,1406.1,386137.3,274.62,6435621.6,4576.93,514849.7,366.15
5,2020,1164.3,313750.3,269.48,5229171.8,4491.26,418333.7,359.30
6,2021,1207.1,316858.4,262.50,5280973.1,4374.93,422477.9,349.99
7,2022,1184.4,313586.7,264.76,5226444.7,4412.74,418115.6,353.02
8,2023,1166.3,306368.2,262.68,5106136.5,4378.06,408490.9,350.25
9,2024,1093.0,285524.7,261.23,4758745.1,4353.84,380699.6,348.31


Q1:
Lauschke GmbH reduced its absolut CO2 emissions by 47.9% between 2015 and 2025. Over the same period, revenue declined by 42%, which raises the of wether the emission reduction is genuine or merely a consequence of shrinking business activity. <br>
The intensity metric provides a clearer picture: intensity declined by 10.3%. <br>
This confirms that the reductions is not purely a revenue effect, production efficiency genuinely improved over the period. However the intensity reduction is significantly smaller than the absolute reduction, indicating that the majority of the emission decline is attributable to reduced business volume. <br>
The same pattern holds for energy consumption and water usage, both of which show consistent intensity reductions alongside absolute decline.

## Trend KPIs

In [4]:
baseline = df_l[df_l["Year"] == 2015].iloc[0]

df_l["CO2_change"] = ((df_l["CarbonEmissions"] - baseline["CarbonEmissions"]) / baseline["CarbonEmissions"] * 100).round(2)
df_l["energy_change"] = ((df_l["EnergyConsumption"] - baseline["EnergyConsumption"]) / baseline["EnergyConsumption"] * 100).round(2)
df_l["water_change"] = ((df_l["WaterUsage"] - baseline["WaterUsage"]) / baseline["WaterUsage"] * 100).round(2)
df_l["ESG_change"] = ((df_l["ESG_Overall"] - baseline["ESG_Overall"]) / baseline["ESG_Overall"] * 100).round(2)

df_l[["Year", "CO2_change", "energy_change", "water_change", "ESG_change"]]

,Year,CO2_change,energy_change,water_change,ESG_change
0,2015,0.00,0.00,0.00,0.00
1,2016,-3.16,-3.16,-3.16,2.32
2,2017,-9.66,-9.66,-9.66,1.93
3,2018,-23.80,-23.80,-23.80,6.56
4,2019,-32.17,-32.17,-32.17,9.27
5,2020,-44.88,-44.88,-44.88,11.78
6,2021,-44.34,-44.34,-44.34,15.83
7,2022,-44.91,-44.91,-44.91,17.37
8,2023,-46.18,-46.18,-46.18,22.01
9,2024,-49.84,-49.84,-49.84,23.17


Dataset-Note: <br>
The change variables seem to be identical which most likely is an error in the synthetic dataset. In a real CSRD context these variable would diverge.

## Benchmarking

In [5]:
bench_yearly = df_bench.groupby("Year").agg(
    CO2_mean = ("CarbonEmissions", "mean"),
    CO2_median = ("CarbonEmissions", "median"),
    ESG_mean = ("ESG_Overall", "mean"),
    ESG_median = ("ESG_Overall", "median"),
    energy_mean = ("EnergyConsumption", "mean"),
    water_mean = ("WaterUsage", "mean"),
    n_companies = ("CompanyID", "count")
).round(2).reset_index()

bench_yearly

,Year,CO2_mean,CO2_median,ESG_mean,ESG_median,energy_mean,water_mean,n_companies
0,2015,337730.99,266378.9,57.77,58.9,5628849.57,450307.95,15
1,2016,345230.88,297841.2,58.34,60.4,5753848.07,460307.85,15
2,2017,352972.67,329135.6,58.60,60.3,5882877.80,470630.24,15
3,2018,358017.36,320722.4,59.13,59.5,5966955.93,477356.47,15
4,2019,371514.49,334947.3,59.98,59.4,6191908.15,495352.64,15
5,2020,367057.69,313750.3,60.51,57.9,6117628.08,489410.25,15
6,2021,365075.07,316858.4,61.41,60.0,6084584.36,486766.75,15
7,2022,377437.74,313586.7,62.19,60.8,6290628.77,503250.31,15
8,2023,392815.13,306368.2,62.59,63.2,6546918.99,523753.52,15
9,2024,399327.29,285524.7,63.42,63.8,6655454.89,532436.39,15


In [6]:
df_comp = df_l[["Year", "CarbonEmissions", "ESG_Overall", "CO2_intensity", "ESG_Environmental", "ESG_Social", "ESG_Governance"]].merge(
    bench_yearly[["Year", "CO2_mean", "ESG_mean"]],
    on="Year"
)

df_comp["ESG_bench"] = (df_comp["ESG_Overall"] - df_comp["ESG_mean"]).round(2)
df_comp["CO2_bench"] = (df_comp["CarbonEmissions"] - df_comp["CO2_mean"]).round(2)

df_comp[["Year", "ESG_Overall", "ESG_mean", "ESG_bench", "CarbonEmissions", "CO2_mean", "CO2_bench"]]

,Year,ESG_Overall,ESG_mean,ESG_bench,CarbonEmissions,CO2_mean,CO2_bench
0,2015,51.8,57.77,-5.97,569254.8,337730.99,231523.81
1,2016,53.0,58.34,-5.34,551293.1,345230.88,206062.22
2,2017,52.8,58.60,-5.80,514255.6,352972.67,161282.93
3,2018,55.2,59.13,-3.93,433770.3,358017.36,75752.94
4,2019,56.6,59.98,-3.38,386137.3,371514.49,14622.81
5,2020,57.9,60.51,-2.61,313750.3,367057.69,-53307.39
6,2021,60.0,61.41,-1.41,316858.4,365075.07,-48216.67
7,2022,60.8,62.19,-1.39,313586.7,377437.74,-63851.04
8,2023,63.2,62.59,0.61,306368.2,392815.13,-86446.93
9,2024,63.8,63.42,0.38,285524.7,399327.29,-113802.59


Q2:
In 2015, Lauschke GmbH ranked below the European manufacturing average on both dimensions ESG score and CO2 emissions above average. <br>
By 2020, Lauschke GmbH crossed below the industry CO2 average for the first time, a milestone reflecting a decade of consistent emission reduction while the industry average rose by 23%.<br>
On ESG scoring, Lauschke GmbH surpassed the industry mean in 2023 and reached above average by 2025, moving from below average to above average performer over the analysis period. <br>
Lauschke GmbH transitioned from a below average performer in 2015 to an above average performer from 2023 to 2025 on both ESG scoring and CO2, emissions relative to European manufacturing peers. 

In [8]:
df_l.to_csv(proc_data_ordner / "lauschke_kpis.csv", index=False)
df_comp.to_csv(proc_data_ordner / "lauschke_benchmark.csv", index=False)
bench_yearly.to_csv(proc_data_ordner / "bench_yearly.csv", index=False)

print("Done")

Done
